In [ ]:
import os
import numpy as np
import pandas as pd
import tifffile as tiff
from Volume_bleeding import simulate_cone_insertion, find_first_black_pixel_slice, process_cone_positions
from Number_bleeding import vessel_seg, process_cone_positions_num


# Choose the files(path)
data_dir = 'mouse_data'          
assert os.path.isdir(data_dir), f"{os.path.abspath(data_dir)} not found - run this notebook from the CF_v2 folder"

# Reslice 2 and 3 are excluded: their raw data is too noisy.
tiff_files = [os.path.join(data_dir, f'Reslice of {i}.tif') for i in [0, 1, 4, 5, 6, 7, 8, 9]]


In [ ]:
# All Parameters Setting

diameters = np.arange(4, 81, 4)  # electrode diameters: 4, 8, 12, ..., 80
y_center = 50                    # insertion postion(y)
length = 1000                    # length of electrodes
pos_num = 100                    # numbers of insertions(for different positions)

In [ ]:
# Segment the vessel for all files

seg_save_dir = "./seg_cache"
os.makedirs(seg_save_dir, exist_ok=True)

for file_path in tiff_files:
    file_label = os.path.splitext(os.path.basename(file_path))[0]
    seg_path = os.path.join(seg_save_dir, f"{file_label}_seg.npz")

    if not os.path.exists(seg_path):
        print(f"Segmenting {file_label} ...")
        img_data = tiff.imread(file_path)
        img_data = np.transpose(img_data, axes=(0, 2, 1)).astype(np.uint16)
        seg_result = vessel_seg(img_data, min_size=2, connectivity=2, distance=2)
        np.savez_compressed(seg_path, seg=seg_result)

    else:
        print(f"Loading cached segmentation: {file_label}")
        seg_result = np.load(seg_path)['seg']


def load_segmented_data(file_path, seg_dir="./seg_cache"):
    file_label = os.path.splitext(os.path.basename(file_path))[0]
    seg_path = os.path.join(seg_dir, f"{file_label}_seg.npz")
    if not os.path.exists(seg_path):
        raise FileNotFoundError(f"Segmentation for {file_label} not found.")
    return np.load(seg_path)['seg']

Loading cached segmentation: Reslice of 0
Loading cached segmentation: Reslice of 1
Loading cached segmentation: Reslice of 4
Loading cached segmentation: Reslice of 5
Loading cached segmentation: Reslice of 6
Loading cached segmentation: Reslice of 7
Loading cached segmentation: Reslice of 8
Loading cached segmentation: Reslice of 9


In [ ]:
# Volume Bleeding per Diameter

save_folder_vol = "Volume_bleeding_per_diameter"  # folder to store CSVs
os.makedirs(save_folder_vol, exist_ok=True)

for file_path in tiff_files:
    print(f"Processing {file_path} ...")
    file_label = os.path.splitext(os.path.basename(file_path))[0]

    # Load
    img_data = tiff.imread(file_path)
    img_data = np.transpose(img_data, axes=(0, 2, 1)).astype(np.uint16)

    # Display the shape of the matrix
    print(f"    Matrix shape: {img_data.shape[0]}(depth, z), {img_data.shape[1]}(height, x), {img_data.shape[2]}(width, y)")

    result_matrix = []
    for diameter in diameters:
        print(f"    Computing diameter: {diameter}")
        radius = diameter / 2

        # Compute area for each insertion x-position (0 to pos_num-1)
        volume = process_cone_positions(
            img_data, y_center=y_center,
            shank_length=length,
            shank_base_diameter=diameter,
            shank_top_diameter=diameter,
            tip_length=0,
            tip_base_diameter=diameter,
            tip_top_diameter=diameter,
            depth_limit=length,
            pos_num=pos_num
        )

        result_matrix.append(volume)

    result_array = np.array(result_matrix).T
    df = pd.DataFrame(result_array, columns=[f"{d}um" for d in diameters])
    df.insert(0, "position", list(range(pos_num)))

    output_csv_path = os.path.join(save_folder_vol, f"{file_label}_Volume.csv")
    df.to_csv(output_csv_path, index=False)



Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 0.tif ...
    Matrix shape: 3000(depth, z), 5000(height, x), 100(width, y)
    Computing diameter: 4
    Counting the volume of intersected vessels...
    Computing diameter: 8
    Counting the volume of intersected vessels...
    Computing diameter: 12
    Counting the volume of intersected vessels...
    Computing diameter: 16
    Counting the volume of intersected vessels...
    Computing diameter: 20
    Counting the volume of intersected vessels...
    Computing diameter: 24
    Counting the volume of intersected vessels...
    Computing diameter: 28
    Counting the volume of intersected vessels...
    Computing diameter: 32
    Counting the volume of intersected vessels...
    Computing diameter: 36
    Counting the volume of intersected vessels...
    Computing diameter: 40
    Counting the volume of intersected vessels...
    Computing diameter: 44
    Counting the volume of intersected vessels...
    Compu

In [ ]:
# Number Bleeding per Diameter

save_folder_num = "Number_bleeding_per_diameter"  # folder to store CSVs
os.makedirs(save_folder_num, exist_ok=True)

for file_path in tiff_files:
    print(f"Processing {file_path} ...")
    file_label = os.path.splitext(os.path.basename(file_path))[0]

    # Segmentation for vessel counting
    img_data_seg = load_segmented_data(file_path)

    # Display the shape of the matrix
    print(f"    Matrix shape: {img_data.shape[0]}(depth, z), {img_data.shape[1]}(height, x), {img_data.shape[2]}(width, y)")

    result_matrix = []
    for diameter in diameters:
        print(f"    Computing diameter: {diameter}")
        radius = diameter / 2

        # Compute area for each insertion x-position (0 to pos_num-1)
        number = process_cone_positions_num(
            img_data_seg, y_center=y_center,
            shank_length=length,
            shank_base_diameter=diameter,
            shank_top_diameter=diameter,
            tip_length=0,
            tip_base_diameter=diameter,
            tip_top_diameter=diameter,
            depth_limit=length,
            pos_num=pos_num
        )

        result_matrix.append(number)

    result_array = np.array(result_matrix).T
    df = pd.DataFrame(result_array, columns=[f"{d}um" for d in diameters])
    df.insert(0, "position", list(range(pos_num)))

    output_csv_path = os.path.join(save_folder_num, f"{file_label}_Number.csv")
    df.to_csv(output_csv_path, index=False)

Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 0.tif ...
    Matrix shape: 3000(depth, z), 5000(height, x), 100(width, y)
    Computing diameter: 4
    Counting the number of intersected vessels...
    Computing diameter: 8
    Counting the number of intersected vessels...
    Computing diameter: 12
    Counting the number of intersected vessels...
    Computing diameter: 16
    Counting the number of intersected vessels...
    Computing diameter: 20
    Counting the number of intersected vessels...
    Computing diameter: 24
    Counting the number of intersected vessels...
    Computing diameter: 28
    Counting the number of intersected vessels...
    Computing diameter: 32
    Counting the number of intersected vessels...
    Computing diameter: 36
    Counting the number of intersected vessels...
    Computing diameter: 40
    Counting the number of intersected vessels...
    Computing diameter: 44
    Counting the number of intersected vessels...
    Compu